## This is a testing notebook for all sorts of quick integration testing

Most tests here should (!) probably be turned in a proper integration test

In [1]:
%load_ext autoreload
%autoreload 2

In [9]:
import hydra

from genpp import BASE_DIR
from genpp.configs import register_resolvers

from genpp.models.layers import CropND

In [3]:
register_resolvers()

In [4]:
with hydra.initialize_config_dir(version_base=None, config_dir=str(BASE_DIR / "configs")):
    cfg = hydra.compose(config_name="base_chen")

In [5]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup("fit")

train_dataloader = datamodule.train_dataloader()

Configuration hash: 05a2d862df574c3ec09a73a2516597b76945436332ad35785c3a5fe80dbc1c8c
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_05a2d862df574c3ec09a73a2516597b76945436332ad35785c3a5fe80dbc1c8c.pt.


In [ ]:
import torch
from tqdm import tqdm
from einops import rearrange


def _fit_scale_variance_td(dl, crop) -> None:
    # Access the training dataloader
    train_loader = dl
    if train_loader is None:
        raise ValueError("Training dataloader is not available.")

    crop_layer = crop
    A = None
    b = None

    with torch.no_grad():
        pbar = tqdm(train_loader, desc="Fitting scale_variance~TD")
        for batch in pbar:
            nwp, obs, td = batch["x"], batch["y"], batch["timedelta"]
            # The NWP vars we need are the first n_vars
            nwp = nwp["predicted_vars"]

            # Initialize A and b
            if A is None:
                A = torch.zeros((nwp.shape[1], 2, 2)).to(nwp)
            if b is None:
                b = torch.zeros((nwp.shape[1], 2)).to(nwp)

            if crop_layer is not None:
                nwp = crop_layer(nwp)

            # If shapes don't match, crop obs as well
            if nwp.shape != obs.shape:
                obs = crop_layer(obs)  # type: ignore

            # Now nwp and obs should have the same shape and no padding
            # We can run the regression now
            td = rearrange(td, "b -> b 1 1 1")
            td = td.expand_as(obs)
            td = rearrange(td, "b c h w -> c (b h w)")
            nwp = rearrange(nwp, "b c h w -> c (b h w)")
            obs = rearrange(obs, "b c h w -> c (b h w)")
            diff = (obs - nwp).abs()
            ones = torch.ones_like(diff)
            X = torch.stack([ones, td], dim=1)
            y = diff

            # The rearrange is the same as calling the transpose on the last two dims if X
            A += torch.bmm(X, rearrange(X, "c a b -> c b a"))
            b += torch.bmm(X, y.unsqueeze(-1)).squeeze(-1)

    betas = torch.linalg.solve(A, b)
    # The first dimension is the variable dimension
    # The second dimension is the intercept and slope
    print(f"Registered buffer with {betas}")
    return betas

In [ ]:
crop = CropND(padding=(0, 1, 1, 2))

In [12]:
betas = _fit_scale_variance_td(train_dataloader, crop)

Fitting scale_variance~TD: 100%|██████████| 171/171 [00:05<00:00, 29.60it/s]

Registered buffer with tensor([[0.0849, 0.0996],
        [0.1960, 0.2183]])
